# RH Unlimited Furniture Delivery — Route Optimization v3
**Hult International Business School | DAT-8564 | Spring 2026**

**v3 — Template-Driven Pipeline:**
- Reads ALL parameters from `RH_Input_Template.xlsx` (SharePoint)
- Config from `Model Config` sheet — no hardcoded constants
- Specs from `Furniture Reference` sheet — no hardcoded item dict
- Orders from `Delivery Orders` sheet — filters by `Status == Active`
- Addresses from `Customer Directory` sheet
- Outputs 4 CSVs for Power BI ingestion
- Designed to run inside Power Query Editor (single Refresh)

In [5]:
# !pip install ortools openpyxl --quiet

## 1. Read Everything from Input Template

In [6]:
import pandas as pd
import numpy as np
import json
from ortools.constraint_solver import routing_enums_pb2, pywrapcp
import warnings
warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════
# SINGLE SOURCE OF TRUTH — the SharePoint template
# ═══════════════════════════════════════════════════════════════
TEMPLATE_PATH = r'C:\Users\jaina\Desktop\BI\A2\RH_Input_Template.xlsx'  # SharePoint path in production

# ── 1a. Model Config ──────────────────────────────────────────
config_df = pd.read_excel(TEMPLATE_PATH, sheet_name='Model Config')
CFG = dict(zip(config_df['parameter'], config_df['value']))

CREW_COST_PER_HOUR = float(CFG['crew_cost_per_hour'])
TRUCK_COST_PER_MILE = float(CFG['operating_cost_per_mile'])
CO2_PER_MILE       = float(CFG['co2_per_mile'])
CO2_IDLE_PER_HOUR  = float(CFG['co2_per_idle_hour'])
TRUCK_MAX_CUFT     = float(CFG['truck_capacity_cuft'])
TRUCK_MAX_LBS      = float(CFG['truck_capacity_lbs'])
DAY_END_MIN        = int(CFG['available_minutes_per_day'])
WAREHOUSE_PREP     = int(CFG['start_of_day_overhead_min'])
STOP_ARRIVE_MIN    = int(CFG['per_stop_arrival_min'])
STOP_DEPART_MIN    = int(CFG['per_stop_departure_min'])
STOP_OVERHEAD_MIN  = STOP_ARRIVE_MIN + STOP_DEPART_MIN
PAY_INTERVAL_MIN   = int(CFG['pay_interval_min'])
HOLDING_SMALL_DAY  = 5.00
HOLDING_LARGE_DAY  = 10.00
WAREHOUSE_ADDRESS  = '590 20th St, San Francisco, CA 94107'

# Labor breakdown from Additional Assumptions
CREW_HOURLY_WAGE    = 25.00
DRIVER_PREMIUM      = 1.20
BENEFITS_MULTIPLIER = 1.30

print(f'Config loaded: crew ${CREW_COST_PER_HOUR}/hr, truck ${TRUCK_COST_PER_MILE}/mi, '
      f'CO₂ {CO2_PER_MILE} kg/mi, capacity {TRUCK_MAX_CUFT} cuft / {TRUCK_MAX_LBS} lbs')

Config loaded: crew $104.0/hr, truck $1.21/mi, CO₂ 1.69 kg/mi, capacity 1700.0 cuft / 7000.0 lbs


In [7]:
# ── 1b. Furniture Reference (replaces hardcoded ITEM_SPECS_EXACT) ──
ref_df = pd.read_excel(TEMPLATE_PATH, sheet_name='Furniture Reference')

# Build lookup: (Brand, Variant) → specs
LARGE_TYPES = {'Dining & Fire Table', 'Sofa', 'Bed Frame', 'Cabinet'}
ref_df['Is_Large'] = ref_df['Item_Type'].isin(LARGE_TYPES)

ITEM_SPECS = {}
for _, r in ref_df.iterrows():
    ITEM_SPECS[(r['Brand'], r['Variant'])] = (
        r['Cu_Ft'], r['Lbs'], r['Load_Min'], r['Unload_Min'], r['Is_Large'], r['Item_Type']
    )

print(f'Loaded {len(ITEM_SPECS)} item specs from Furniture Reference sheet.')

Loaded 39 item specs from Furniture Reference sheet.


In [8]:
# ── 1c. Delivery Orders — ACTIVE only ─────────────────────────
orders_raw = pd.read_excel(TEMPLATE_PATH, sheet_name='Delivery Orders')

# Rename template columns to internal names
orders_raw = orders_raw.rename(columns={
    'Line #':           'Line',
    'Status  \u270f\ufe0f':       'Status',
    'Reschedule Date  \u270f\ufe0f': 'Reschedule_Date',
    'Manager Notes  \u270f\ufe0f':  'Manager_Notes',
    'Size / Variant':   'Product Description',
    'Product':          'Brand',
    'Qty':              'Quantity',
})

# Filter active orders only
orders_df = orders_raw[orders_raw['Status'] == 'Active'].copy()
print(f'Active orders: {len(orders_df)} lines (of {len(orders_raw)} total)')

# ── 1d. Customer Directory ────────────────────────────────────
cust_dir = pd.read_excel(TEMPLATE_PATH, sheet_name='Customer Directory')
addresses = dict(zip(cust_dir['Customer ID'], cust_dir['Full Address']))

# Active customers only
customers = sorted(orders_df['Customer ID'].unique().tolist())
print(f'Active customers: {len(customers)}')
for c in customers:
    print(f'  {c}: {addresses[c]}')

Active orders: 139 lines (of 139 total)
Active customers: 17
  CUST1837: 535 Bridgeway, Sausalito, CA 94965
  CUST2663: 1520 165th Ave, San Leandro, CA 94578
  CUST3465: 840 Oak Street, Alameda, CA 94501
  CUST5158: 13366 Campus Dr, Oakland, CA 94619
  CUST5179: 456 Sentinel Drive, Alameda, CA 94501
  CUST5385: 1500 La Loma Ave, Berkeley, CA 94708
  CUST6288: 1121 40th St APT 4206, Emeryville, CA 94608
  CUST6648: 1080 Chestnut St UNIT 8A, San Francisco, CA 94109
  CUST6796: 900 Spruce St, Berkeley, CA 94707
  CUST6958: 1201 California St UNIT 602, San Francisco, CA 94109
  CUST7120: 1550 165th Ave, San Leandro, CA 94578
  CUST7706: 5 Drury Road, Berkeley, CA 94705
  CUST8585: 1750 Taylor St UNIT 1401, San Francisco, CA 94133
  CUST8614: 8 Admiral Dr Unit A232, Emeryville, CA 94608
  CUST9283: 13367 Campus Drive, Oakland, CA 94619
  CUST9602: 13427 Campus Drive, Oakland, CA 94619
  CUST9615: 595 Muscari St, San Leandro, CA 94578


## 2. Apply Specs from Furniture Reference

In [9]:
def apply_specs(row):
    key = (row['Brand'], row['Product Description'])
    specs = ITEM_SPECS.get(key)
    if specs:
        cu_ft, lbs, load_min, unload_min, is_large, item_type = specs
        return pd.Series({
            'CuFt':       cu_ft      * row['Quantity'],
            'Lbs':        lbs        * row['Quantity'],
            'Load Min':   load_min   * row['Quantity'],
            'Unload Min': unload_min * row['Quantity'],
            'Is Large':   is_large,
            'Category':   item_type,
        })
    else:
        print(f'  ⚠ No match: {key}')
        return pd.Series({'CuFt': 0, 'Lbs': 0, 'Load Min': 0, 'Unload Min': 0,
                          'Is Large': False, 'Category': 'Unknown'})

specs_cols = orders_df.apply(apply_specs, axis=1)
orders_df  = pd.concat([orders_df.reset_index(drop=True), specs_cols], axis=1)

# Per-customer summary
customer_summary = orders_df.groupby('Customer ID').agg(
    total_load_min   = ('Load Min',   'sum'),
    total_unload_min = ('Unload Min', 'sum'),
    total_cuft       = ('CuFt',       'sum'),
    total_lbs        = ('Lbs',        'sum'),
    large_items      = ('Is Large',   'sum'),
    small_items      = ('Is Large',   lambda x: (~x).sum()),
).reset_index()

customer_summary['stop_overhead_min'] = STOP_OVERHEAD_MIN
customer_summary['total_stop_min']    = (
    customer_summary['total_unload_min'] + customer_summary['stop_overhead_min']
)

print(customer_summary[['Customer ID','total_cuft','total_lbs','total_stop_min']].to_string(index=False))
print(f'\nTotal volume: {customer_summary["total_cuft"].sum():.0f} cu ft')
print(f'Min truckloads needed: {int(np.ceil(customer_summary["total_cuft"].sum() / TRUCK_MAX_CUFT))}')

oversized = customer_summary[
    (customer_summary['total_cuft'] > TRUCK_MAX_CUFT) |
    (customer_summary['total_lbs']  > TRUCK_MAX_LBS)
]
if len(oversized):
    print('\nCustomers needing split loads:')
    for _, r in oversized.iterrows():
        print(f"  {r['Customer ID']}: {r['total_cuft']:.0f} cuft, {r['total_lbs']:.0f} lbs")

Customer ID  total_cuft  total_lbs  total_stop_min
   CUST1837     1893.94       4616           125.0
   CUST2663       68.50        761            55.0
   CUST3465      205.74        687            48.0
   CUST5158      600.70       1332            62.5
   CUST5179     1867.26       4429           132.5
   CUST5385      224.36        449            22.0
   CUST6288     1433.41       2644            86.0
   CUST6648     1307.62       5921            97.5
   CUST6796      968.89       3518            92.0
   CUST6958     1340.38       3610            83.0
   CUST7120      976.74       4401           149.5
   CUST7706       68.42        370            27.0
   CUST8585      254.06        851            61.0
   CUST8614     1549.97       7063           142.0
   CUST9283       61.05        121            27.0
   CUST9602      241.04       1629            30.0
   CUST9615     1535.54       4414           103.0

Total volume: 14598 cu ft
Min truckloads needed: 9

Customers needing split loads

## 3. Split Oversized Orders & Build Delivery Nodes

In [10]:
def split_customer_into_loads(customer_id):
    items = orders_df[orders_df['Customer ID'] == customer_id].copy()
    items = items.sort_values('CuFt', ascending=False)
    loads, cur_load, cur_cuft, cur_lbs = [], [], 0, 0

    for _, item in items.iterrows():
        unit_cuft = item['CuFt'] / item['Quantity']
        unit_lbs  = item['Lbs']  / item['Quantity']
        for _ in range(int(item['Quantity'])):
            if cur_cuft + unit_cuft <= TRUCK_MAX_CUFT and cur_lbs + unit_lbs <= TRUCK_MAX_LBS:
                cur_load.append(item)
                cur_cuft += unit_cuft
                cur_lbs  += unit_lbs
            else:
                loads.append({'items': cur_load, 'cuft': cur_cuft, 'lbs': cur_lbs})
                cur_load = [item]
                cur_cuft = unit_cuft
                cur_lbs  = unit_lbs
    if cur_load:
        loads.append({'items': cur_load, 'cuft': cur_cuft, 'lbs': cur_lbs})
    return loads

delivery_nodes = []
for cid in customers:
    row = customer_summary[customer_summary['Customer ID'] == cid].iloc[0]
    if row['total_cuft'] <= TRUCK_MAX_CUFT and row['total_lbs'] <= TRUCK_MAX_LBS:
        delivery_nodes.append({
            'node_id': cid, 'customer_id': cid, 'load_num': 1, 'total_loads': 1,
            'cuft': row['total_cuft'], 'lbs': row['total_lbs'],
            'total_load_min': row['total_load_min'],
            'total_stop_min': row['total_stop_min'],
            'large_items': row['large_items'], 'small_items': row['small_items'],
        })
    else:
        loads = split_customer_into_loads(cid)
        print(f'  {cid} → {len(loads)} loads')
        for i, load in enumerate(loads, 1):
            load_load_min   = sum(it['Load Min'] / it['Quantity'] for it in load['items'])
            load_unload_min = sum(it['Unload Min'] / it['Quantity'] for it in load['items'])
            large = sum(1 for it in load['items'] if it['Is Large'])
            small = sum(1 for it in load['items'] if not it['Is Large'])
            print(f'    L{i}: {load["cuft"]:.0f} cuft, {load["lbs"]:.0f} lbs')
            delivery_nodes.append({
                'node_id': f'{cid}_L{i}', 'customer_id': cid,
                'load_num': i, 'total_loads': len(loads),
                'cuft': load['cuft'], 'lbs': load['lbs'],
                'total_load_min': load_load_min,
                'total_stop_min': load_unload_min + STOP_OVERHEAD_MIN,
                'large_items': large, 'small_items': small,
            })

nodes_df = pd.DataFrame(delivery_nodes)
print(f'\nTotal delivery nodes: {len(nodes_df)}')
print(nodes_df[['node_id','cuft','lbs','total_stop_min']].to_string(index=False))

  CUST1837 → 2 loads
    L1: 1668 cuft, 3624 lbs
    L2: 226 cuft, 992 lbs
  CUST5179 → 2 loads
    L1: 1694 cuft, 3590 lbs
    L2: 173 cuft, 839 lbs
  CUST8614 → 2 loads
    L1: 1530 cuft, 6982 lbs
    L2: 20 cuft, 81 lbs

Total delivery nodes: 20
    node_id    cuft    lbs  total_stop_min
CUST1837_L1 1668.02 3624.0            74.0
CUST1837_L2  225.92  992.0            65.0
   CUST2663   68.50  761.0            55.0
   CUST3465  205.74  687.0            48.0
   CUST5158  600.70 1332.0            62.5
CUST5179_L1 1694.11 3590.0            70.0
CUST5179_L2  173.15  839.0            76.5
   CUST5385  224.36  449.0            22.0
   CUST6288 1433.41 2644.0            86.0
   CUST6648 1307.62 5921.0            97.5
   CUST6796  968.89 3518.0            92.0
   CUST6958 1340.38 3610.0            83.0
   CUST7120  976.74 4401.0           149.5
   CUST7706   68.42  370.0            27.0
   CUST8585  254.06  851.0            61.0
CUST8614_L1 1530.25 6982.0           126.0
CUST8614_L2   19.72 

## 4. Distance & Duration Matrix (hardcoded from Google Maps API)

In [11]:
# Row/col order: [warehouse, CUST1837, CUST2663, CUST3465, CUST5158, CUST5179,
#                 CUST5385, CUST6288, CUST6648, CUST6796, CUST6958, CUST7120,
#                 CUST7706, CUST8585, CUST8614, CUST9283, CUST9602, CUST9615]
# NOTE: customers list is SORTED alphabetically.
# The original matrix used a different order. We must remap.

# Original matrix order:
ORIG_ORDER = ['warehouse', 'CUST5179', 'CUST1837', 'CUST8614', 'CUST3465',
              'CUST6796', 'CUST7120', 'CUST6958', 'CUST8585', 'CUST9615',
              'CUST6288', 'CUST6648', 'CUST5158', 'CUST9283', 'CUST7706',
              'CUST9602', 'CUST2663', 'CUST5385']

_dist_orig = [
    [0,    8.2,  10.5,  3.1,  8.4,  6.3, 18.2,  2.8,  2.9, 17.8,  3.2,  2.5, 15.1, 15.1,  7.1, 15.3, 18.0,  7.8],
    [8.2,  0,    16.0,  4.5,  0.4,  8.5, 14.5, 10.1, 10.3, 14.1,  4.8,  9.8, 11.4, 11.4,  9.8, 11.6, 14.3, 10.5],
    [10.5, 16.0, 0,    12.1, 16.2,  8.9, 25.0,  8.3,  7.9, 24.6, 11.5,  8.0, 22.0, 22.0,  7.5, 22.2, 24.8,  9.6],
    [3.1,  4.5,  12.1,  0,    4.9,  4.8, 15.8,  5.4,  5.6, 15.4,  0.4,  5.1, 12.3, 12.3,  5.8, 12.5, 15.6,  6.5],
    [8.4,  0.4,  16.2,  4.9,  0,    8.7, 14.7, 10.3, 10.5, 14.3,  5.0, 10.0, 11.6, 11.6, 10.0, 11.8, 14.5, 10.7],
    [6.3,  8.5,   8.9,  4.8,  8.7,  0,  20.5,  4.2,  4.0, 20.1,  4.5,  3.9, 17.4, 17.4,  1.8, 17.6, 20.3,  2.5],
    [18.2, 14.5, 25.0, 15.8, 14.7, 20.5,  0,   20.0, 20.2,  0.4, 15.3, 19.7,  3.2,  3.2, 21.2,  2.9,  0.6, 22.0],
    [2.8,  10.1,  8.3,  5.4, 10.3,  4.2, 20.0,  0,    0.5, 19.6,  4.9,  0.3, 16.9, 16.9,  5.5, 17.1, 19.8,  6.2],
    [2.9,  10.3,  7.9,  5.6, 10.5,  4.0, 20.2,  0.5,  0,  19.8,  5.1,  0.2, 17.1, 17.1,  5.3, 17.3, 20.0,  6.0],
    [17.8, 14.1, 24.6, 15.4, 14.3, 20.1,  0.4, 19.6, 19.8,  0,  15.0, 19.4,  2.8,  2.8, 20.8,  2.5,  0.2, 21.6],
    [3.2,  4.8,  11.5,  0.4,  5.0,  4.5, 15.3,  4.9,  5.1, 15.0,  0,    4.8, 12.2, 12.2,  5.3, 12.4, 15.1,  6.0],
    [2.5,  9.8,   8.0,  5.1, 10.0,  3.9, 19.7,  0.3,  0.2, 19.4,  4.8,  0,  16.6, 16.6,  5.2, 16.8, 19.5,  5.9],
    [15.1, 11.4, 22.0, 12.3, 11.6, 17.4,  3.2, 16.9, 17.1,  2.8, 12.2, 16.6,  0,    0.1, 18.1,  0.3,  2.8, 18.8],
    [15.1, 11.4, 22.0, 12.3, 11.6, 17.4,  3.2, 16.9, 17.1,  2.8, 12.2, 16.6,  0.1,  0,  18.1,  0.3,  2.8, 18.8],
    [7.1,   9.8,  7.5,  5.8, 10.0,  1.8, 21.2,  5.5,  5.3, 20.8,  5.3,  5.2, 18.1, 18.1,  0,  18.3, 21.0,  1.5],
    [15.3, 11.6, 22.2, 12.5, 11.8, 17.6,  2.9, 17.1, 17.3,  2.5, 12.4, 16.8,  0.3,  0.3, 18.3,  0,    2.5, 19.0],
    [18.0, 14.3, 24.8, 15.6, 14.5, 20.3,  0.6, 19.8, 20.0,  0.2, 15.1, 19.5,  2.8,  2.8, 21.0,  2.5,  0,  21.8],
    [7.8,  10.5,  9.6,  6.5, 10.7,  2.5, 22.0,  6.2,  6.0, 21.6,  6.0,  5.9, 18.8, 18.8,  1.5, 19.0, 21.8,  0],
]

_dur_orig = [
    [0,   22,  28,  14,  22,  18,  38,  15,  15,  37,  15,  14,  32,  32,  20,  33,  38,  22],
    [22,   0,  38,  18,   8,  24,  35,  28,  28,  35,  16,  27,  29,  29,  26,  30,  35,  28],
    [28,  38,   0,  30,  38,  22,  50,  25,  24,  50,  28,  24,  44,  44,  22,  45,  50,  25],
    [14,  18,  30,   0,  18,  16,  34,  18,  18,  34,   8,  17,  28,  28,  18,  29,  34,  20],
    [22,   8,  38,  18,   0,  24,  36,  28,  28,  36,  16,  27,  30,  30,  26,  31,  36,  28],
    [18,  24,  22,  16,  24,   0,  42,  18,  17,  41,  16,  16,  36,  36,  10,  37,  42,  12],
    [38,  35,  50,  34,  36,  42,   0,  40,  40,   8,  34,  39,  14,  14,  44,  12,   8,  46],
    [15,  28,  25,  18,  28,  18,  40,   0,   5,  39,  17,   4,  34,  34,  22,  35,  40,  24],
    [15,  28,  24,  18,  28,  17,  40,   5,   0,  39,  17,   3,  34,  34,  21,  35,  40,  23],
    [37,  35,  50,  34,  36,  41,   8,  39,  39,   0,  34,  38,  13,  13,  43,  11,   6,  45],
    [15,  16,  28,   8,  16,  16,  34,  17,  17,  34,   0,  16,  28,  28,  18,  29,  34,  20],
    [14,  27,  24,  17,  27,  16,  39,   4,   3,  38,  16,   0,  33,  33,  20,  34,  39,  22],
    [32,  29,  44,  28,  30,  36,  14,  34,  34,  13,  28,  33,   0,   3,  38,   4,   9,  40],
    [32,  29,  44,  28,  30,  36,  14,  34,  34,  13,  28,  33,   3,   0,  38,   4,   9,  40],
    [20,  26,  22,  18,  26,  10,  44,  22,  21,  43,  18,  20,  38,  38,   0,  39,  44,   8],
    [33,  30,  45,  29,  31,  37,  12,  35,  35,  11,  29,  34,   4,   4,  39,   0,   7,  41],
    [38,  35,  50,  34,  36,  42,   8,  40,  40,   6,  34,  39,   9,   9,  44,   7,   0,  46],
    [22,  28,  25,  20,  28,  12,  46,  24,  23,  45,  20,  22,  40,  40,   8,  41,  46,   0],
]

# Remap to sorted customer order: [warehouse] + sorted(customers)
NEW_ORDER = ['warehouse'] + customers
remap = [ORIG_ORDER.index(x) for x in NEW_ORDER]

n = len(NEW_ORDER)
distance_matrix_miles = [[_dist_orig[remap[i]][remap[j]] for j in range(n)] for i in range(n)]
duration_matrix_min   = [[_dur_orig[remap[i]][remap[j]]  for j in range(n)] for i in range(n)]
distance_matrix_int   = [[int(v * 1000) for v in row] for row in distance_matrix_miles]
duration_matrix_int   = [[int(v) for v in row] for row in duration_matrix_min]

def get_travel(from_cid, to_cid, mode='time'):
    fi = 0 if from_cid == 'warehouse' else customers.index(from_cid) + 1
    ti = 0 if to_cid   == 'warehouse' else customers.index(to_cid)   + 1
    return duration_matrix_min[fi][ti] if mode == 'time' else distance_matrix_miles[fi][ti]

print(f'Matrix: {n}×{n} (sorted order: {NEW_ORDER[1]}, {NEW_ORDER[2]}, ... {NEW_ORDER[-1]})')
print(f'Warehouse → {customers[0]}: {get_travel("warehouse", customers[0], "miles"):.1f} mi, '
      f'{get_travel("warehouse", customers[0], "time"):.0f} min')

Matrix: 18×18 (sorted order: CUST1837, CUST2663, ... CUST9615)
Warehouse → CUST1837: 10.5 mi, 28 min


## 5. Cost & CO₂ Functions

In [12]:
def round_to_quarter_hour(hours):
    minutes   = hours * 60
    remainder = minutes % PAY_INTERVAL_MIN
    threshold = PAY_INTERVAL_MIN / 2
    rounded   = minutes + (PAY_INTERVAL_MIN - remainder) if remainder >= threshold else minutes - remainder
    return rounded / 60

def compute_labor_cost(total_hours_raw):
    hours        = round_to_quarter_hour(total_hours_raw)
    standard_pay = 2 * CREW_HOURLY_WAGE * hours
    driver_pay   = CREW_HOURLY_WAGE * DRIVER_PREMIUM * hours
    return round((standard_pay + driver_pay) * BENEFITS_MULTIPLIER, 2)

def compute_truck_cost(total_miles):
    return round(total_miles * TRUCK_COST_PER_MILE, 2)

def compute_holding_cost(customer_id, day_number):
    if day_number <= 1:
        return 0
    days_waiting = day_number - 1
    cust_items   = orders_df[orders_df['Customer ID'] == customer_id]
    large = cust_items[cust_items['Is Large']]['Quantity'].sum()
    small = cust_items[~cust_items['Is Large']]['Quantity'].sum()
    return round((large * HOLDING_LARGE_DAY + small * HOLDING_SMALL_DAY) * days_waiting, 2)

def compute_co2(total_miles, idle_hours):
    return round(total_miles * CO2_PER_MILE + idle_hours * CO2_IDLE_PER_HOUR, 2)

print(f'Example 9hr day, 80mi, 2hr idle:')
print(f'  Labor: ${compute_labor_cost(9):.2f} | Truck: ${compute_truck_cost(80):.2f} | CO₂: {compute_co2(80,2):.1f} kg')

Example 9hr day, 80mi, 2hr idle:
  Labor: $936.00 | Truck: $96.80 | CO₂: 143.1 kg


## 6. OR-Tools Route Solver

In [13]:
def solve_daily_route(node_ids_today, day_number=1):
    if not node_ids_today:
        return []

    def node_to_matrix_idx(nid):
        cid = nodes_df[nodes_df['node_id'] == nid]['customer_id'].values[0]
        return customers.index(cid) + 1

    matrix_indices = [0] + [node_to_matrix_idx(nid) for nid in node_ids_today]
    m = len(matrix_indices)

    dist_sub = [[distance_matrix_int[matrix_indices[i]][matrix_indices[j]] for j in range(m)] for i in range(m)]
    time_sub = [[duration_matrix_int[matrix_indices[i]][matrix_indices[j]] for j in range(m)] for i in range(m)]

    service_times = [0]
    for nid in node_ids_today:
        row = nodes_df[nodes_df['node_id'] == nid].iloc[0]
        service_times.append(int(row['total_stop_min']))

    demands_cuft = [0] + [int(nodes_df[nodes_df['node_id']==nid]['cuft'].values[0] * 10) for nid in node_ids_today]
    demands_lbs  = [0] + [int(nodes_df[nodes_df['node_id']==nid]['lbs'].values[0])        for nid in node_ids_today]

    manager = pywrapcp.RoutingIndexManager(m, 1, 0)
    routing = pywrapcp.RoutingModel(manager)

    def distance_cb(from_idx, to_idx):
        return dist_sub[manager.IndexToNode(from_idx)][manager.IndexToNode(to_idx)]
    routing.SetArcCostEvaluatorOfAllVehicles(routing.RegisterTransitCallback(distance_cb))

    def time_cb(from_idx, to_idx):
        fn = manager.IndexToNode(from_idx)
        tn = manager.IndexToNode(to_idx)
        return time_sub[fn][tn] + service_times[fn]
    time_cb_idx = routing.RegisterTransitCallback(time_cb)

    routing.AddDimension(time_cb_idx, 60, DAY_END_MIN, False, 'Time')
    time_dim = routing.GetDimensionOrDie('Time')
    time_dim.CumulVar(routing.Start(0)).SetRange(WAREHOUSE_PREP, DAY_END_MIN)
    for i in range(1, m):
        time_dim.CumulVar(manager.NodeToIndex(i)).SetRange(0, DAY_END_MIN)

    def cuft_cb(idx):
        return demands_cuft[manager.IndexToNode(idx)]
    routing.AddDimensionWithVehicleCapacity(
        routing.RegisterUnaryTransitCallback(cuft_cb), 0, [int(TRUCK_MAX_CUFT * 10)], True, 'Volume')

    def lbs_cb(idx):
        return demands_lbs[manager.IndexToNode(idx)]
    routing.AddDimensionWithVehicleCapacity(
        routing.RegisterUnaryTransitCallback(lbs_cb), 0, [int(TRUCK_MAX_LBS)], True, 'Weight')

    search_params = pywrapcp.DefaultRoutingSearchParameters()
    search_params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    search_params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search_params.time_limit.seconds = 30

    solution = routing.SolveWithParameters(search_params)
    if not solution:
        print(f'  ⚠ No solution for day {day_number}')
        return node_ids_today  # fallback: original order

    route = []
    index = routing.Start(0)
    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        if node > 0:
            route.append(node_ids_today[node - 1])
        index = solution.Value(routing.NextVar(index))
    return route

print('OR-Tools solver ready.')

OR-Tools solver ready.


## 7. Scheduler — Bin-Pack + Time Feasibility

In [14]:
def build_full_schedule():
    remaining = nodes_df.sort_values('cuft', ascending=False)['node_id'].tolist()
    schedule  = []
    day_num   = 1
    day_names = ['Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']

    while remaining:
        day_label = day_names[(day_num - 1) % 5]
        today, used_cuft, used_lbs, used_time = [], 0, 0, WAREHOUSE_PREP

        for nid in remaining[:]:
            node_row = nodes_df[nodes_df['node_id'] == nid].iloc[0]
            c, w = node_row['cuft'], node_row['lbs']

            if used_cuft + c > TRUCK_MAX_CUFT or used_lbs + w > TRUCK_MAX_LBS:
                continue

            this_cid = node_row['customer_id']
            if today:
                last_cid  = nodes_df[nodes_df['node_id'] == today[-1]]['customer_id'].values[0]
                travel_to = get_travel(last_cid, this_cid, 'time')
            else:
                travel_to = get_travel('warehouse', this_cid, 'time')

            service    = node_row['total_stop_min']
            return_min = get_travel(this_cid, 'warehouse', 'time')

            if used_time + travel_to + service + return_min > DAY_END_MIN:
                continue

            today.append(nid)
            used_cuft += c
            used_lbs  += w
            used_time += travel_to + service

        if not today:
            print(f'WARNING: No nodes fit on Day {day_num}. Remaining: {remaining[:3]}')
            break

        for nid in today:
            remaining.remove(nid)

        route = solve_daily_route(today, day_number=day_num)
        print(f'Day {day_num} ({day_label}): {len(today)} stops | {used_cuft:.0f} cuft | route: {route}')

        schedule.append({
            'day': day_num, 'day_label': day_label,
            'nodes': today, 'route': route,
            'cuft': used_cuft, 'lbs': used_lbs,
        })
        day_num += 1

    return schedule

schedule = build_full_schedule()
print(f'\nTotal delivery days: {len(schedule)}')

Day 1 (Tuesday): 1 stops | 1694 cuft | route: ['CUST5179_L1']
Day 2 (Wednesday): 2 stops | 1688 cuft | route: ['CUST8614_L2', 'CUST1837_L1']
Day 3 (Thursday): 3 stops | 1672 cuft | route: ['CUST7706', 'CUST9615', 'CUST2663']
Day 4 (Friday): 1 stops | 1530 cuft | route: ['CUST8614_L1']
Day 5 (Saturday): 2 stops | 1687 cuft | route: ['CUST8585', 'CUST6288']
Day 6 (Tuesday): 3 stops | 1642 cuft | route: ['CUST6958', 'CUST9283', 'CUST9602']
Day 7 (Wednesday): 2 stops | 1534 cuft | route: ['CUST6648', 'CUST1837_L2']
Day 8 (Thursday): 2 stops | 1577 cuft | route: ['CUST5158', 'CUST7120']
Day 9 (Friday): 4 stops | 1572 cuft | route: ['CUST5385', 'CUST6796', 'CUST5179_L2', 'CUST3465']

Total delivery days: 9


## 8. Generate Per-Day Detail

In [15]:
def generate_day_detail(day_entry):
    route, day_num = day_entry['route'], day_entry['day']
    if not route:
        return None

    current_time, prev_node = WAREHOUSE_PREP, 0
    total_miles, total_idle_h = 0, 0
    stops = []

    for nid in route:
        node_row    = nodes_df[nodes_df['node_id'] == nid].iloc[0]
        cid         = node_row['customer_id']
        matrix_idx  = customers.index(cid) + 1
        travel_min  = duration_matrix_int[prev_node][matrix_idx]
        service_min = int(node_row['total_stop_min'])
        miles       = distance_matrix_miles[prev_node][matrix_idx]

        arrive_min  = current_time + travel_min
        depart_min  = arrive_min + service_min
        total_miles  += miles
        total_idle_h += service_min / 60

        load_label = f' (Load {int(node_row["load_num"])} of {int(node_row["total_loads"])})' if node_row['total_loads'] > 1 else ''

        stops.append({
            'node_id': nid, 'customer_id': cid, 'load_label': load_label,
            'address': addresses[cid],
            'arrive': f'{8 + arrive_min // 60}:{arrive_min % 60:02d}',
            'depart': f'{8 + depart_min // 60}:{depart_min % 60:02d}',
            'travel_min': travel_min, 'service_min': service_min, 'miles': round(miles, 2),
        })
        current_time = depart_min
        prev_node    = matrix_idx

    total_miles += distance_matrix_miles[prev_node][0]
    total_hours  = current_time / 60
    total_load_min = sum(nodes_df[nodes_df['node_id']==nid]['total_load_min'].values[0] for nid in route)
    total_idle_h += total_load_min / 60

    unique_cids = list(set(nodes_df[nodes_df['node_id'].isin(route)]['customer_id'].tolist()))
    labor_cost   = compute_labor_cost(total_hours)
    truck_cost   = compute_truck_cost(total_miles)
    holding_cost = sum(compute_holding_cost(cid, day_num) for cid in unique_cids)

    return {
        'day': day_num, 'day_label': day_entry['day_label'], 'stops': stops,
        'total_miles': round(total_miles, 2), 'total_hours': round(total_hours, 2),
        'labor_cost': labor_cost, 'truck_cost': truck_cost, 'holding_cost': holding_cost,
        'total_cost': labor_cost + truck_cost + holding_cost,
        'co2_kg': compute_co2(total_miles, total_idle_h),
    }

daily_details = [d for d in (generate_day_detail(s) for s in schedule) if d]
print(f'Details generated for {len(daily_details)} days.')

Details generated for 9 days.


## 9. Schedule Report

In [16]:
cum_cost = cum_co2 = 0
for d in daily_details:
    cum_cost += d['total_cost']
    cum_co2  += d['co2_kg']
    print(f"{'='*60}")
    print(f"Day {d['day']} — {d['day_label']}")
    print(f"{'='*60}")
    for i, s in enumerate(d['stops'], 1):
        print(f"  Stop {i}: {s['customer_id']}{s['load_label']}")
        print(f"    {s['address']}")
        print(f"    Arrive: {s['arrive']}  Depart: {s['depart']}  ({s['miles']} mi)")
    print(f"  Miles: {d['total_miles']}  Hours: {d['total_hours']}")
    print(f"  Labor: ${d['labor_cost']:.2f}  Truck: ${d['truck_cost']:.2f}  Holding: ${d['holding_cost']:.2f}")
    print(f"  TOTAL: ${d['total_cost']:.2f}  CO₂: {d['co2_kg']:.1f} kg")
    print(f"  Cumulative: ${cum_cost:.2f}  CO₂: {cum_co2:.1f} kg\n")

Day 1 — Tuesday
  Stop 1: CUST5179 (Load 1 of 2)
    456 Sentinel Drive, Alameda, CA 94501
    Arrive: 8:45  Depart: 9:55  (8.2 mi)
  Miles: 16.4  Hours: 1.92
  Labor: $208.00  Truck: $19.84  Holding: $0.00
  TOTAL: $227.84  CO₂: 35.0 kg
  Cumulative: $227.84  CO₂: 35.0 kg

Day 2 — Wednesday
  Stop 1: CUST8614 (Load 2 of 2)
    8 Admiral Dr Unit A232, Emeryville, CA 94608
    Arrive: 8:37  Depart: 9:07  (3.1 mi)
  Stop 2: CUST1837 (Load 1 of 2)
    535 Bridgeway, Sausalito, CA 94965
    Arrive: 9:37  Depart: 10:51  (12.1 mi)
  Miles: 25.7  Hours: 2.85
  Labor: $286.00  Truck: $31.10  Holding: $455.00
  TOTAL: $772.10  CO₂: 53.7 kg
  Cumulative: $999.94  CO₂: 88.7 kg

Day 3 — Thursday
  Stop 1: CUST7706
    5 Drury Road, Berkeley, CA 94705
    Arrive: 8:43  Depart: 9:10  (7.1 mi)
  Stop 2: CUST9615
    595 Muscari St, San Leandro, CA 94578
    Arrive: 9:53  Depart: 11:36  (20.8 mi)
  Stop 3: CUST2663
    1520 165th Ave, San Leandro, CA 94578
    Arrive: 11:42  Depart: 12:37  (0.2 mi)
  

## 10. Loading Manifest

In [17]:
for d in daily_details:
    print(f"\nDay {d['day']} — {d['day_label']} LOADING MANIFEST")
    print('First item = deepest in truck (last delivery loaded first)')
    print('-' * 60)
    for pos, s in enumerate(reversed(d['stops']), 1):
        cid = s['customer_id']
        print(f"  [{pos}] {cid}{s['load_label']} — {s['address']}")
        items = orders_df[orders_df['Customer ID'] == cid][['Product Description','Category','Quantity']]
        for _, item in items.iterrows():
            print(f"       • {item['Quantity']}× {item['Product Description']} ({item['Category']})")


Day 1 — Tuesday LOADING MANIFEST
First item = deepest in truck (last delivery loaded first)
------------------------------------------------------------
  [1] CUST5179 (Load 1 of 2) — 456 Sentinel Drive, Alameda, CA 94501
       • 1× 36'' Bistro Table (Dining & Fire Table)
       • 4× Dining Chair (Chair)
       • 1× 42'' Bistro Table (Dining & Fire Table)
       • 4× Dining Chair (Chair)
       • 3× Classic 72'' Sectional (Sofa)
       • 2× Dining Side Chair (Chair)
       • 2× King Bed 102'' (Bed Frame)
       • 3× 32'' Nightstand (Coffee Table & Nightstand)
       • 1× 56'' W Fire Table (Dining & Fire Table)
       • 3× Cal King Bed (Bed Frame)
       • 2× Carbon Fountain (Cabinets and Fountains)

Day 2 — Wednesday LOADING MANIFEST
First item = deepest in truck (last delivery loaded first)
------------------------------------------------------------
  [1] CUST1837 (Load 1 of 2) — 535 Bridgeway, Sausalito, CA 94965
       • 1× 30'' Bistro Table (Dining & Fire Table)
       • 4× Dini

## 11. CO₂ Comparison — Optimized vs Naive Baseline

In [18]:
def compute_naive_co2():
    total_miles = total_idle = 0
    for d in daily_details:
        delivered = [s['customer_id'] for s in d['stops']]
        naive_order = [cid for cid in customers if cid in delivered]
        prev_node = 0
        for cid in naive_order:
            node = customers.index(cid) + 1
            total_miles += distance_matrix_miles[prev_node][node]
            total_idle  += nodes_df[nodes_df['customer_id'] == cid].iloc[0]['total_stop_min'] / 60
            prev_node = node
        total_miles += distance_matrix_miles[prev_node][0]
    return compute_co2(total_miles, total_idle), round(total_miles, 2)

naive_co2, naive_miles = compute_naive_co2()
opt_co2   = sum(d['co2_kg']      for d in daily_details)
opt_miles = sum(d['total_miles'] for d in daily_details)

print('CO₂ COMPARISON')
print('─' * 45)
print(f'  Naive:     {naive_miles:.1f} mi  |  {naive_co2:.1f} kg CO₂')
print(f'  Optimized: {opt_miles:.1f} mi  |  {opt_co2:.1f} kg CO₂')
print(f'  Saved:     {naive_miles - opt_miles:.1f} mi  |  {naive_co2 - opt_co2:.1f} kg CO₂')
print(f'  Improvement: {((naive_co2 - opt_co2)/naive_co2*100):.1f}%')

CO₂ COMPARISON
─────────────────────────────────────────────
  Naive:     258.0 mi  |  534.2 kg CO₂
  Optimized: 226.0 mi  |  515.8 kg CO₂
  Saved:     32.0 mi  |  18.4 kg CO₂
  Improvement: 3.4%


## 12. Export CSVs for Power BI

In [19]:
# ── Schedule ──────────────────────────────────────────────────
schedule_rows = []
for d in daily_details:
    for i, s in enumerate(d['stops'], 1):
        schedule_rows.append({
            'Day': d['day'], 'Day_Label': d['day_label'], 'Stop_Order': i,
            'Node_ID': s['node_id'], 'Customer_ID': s['customer_id'],
            'Load_Label': s['load_label'], 'Address': s['address'],
            'Arrive': s['arrive'], 'Depart': s['depart'],
            'Travel_Min': s['travel_min'], 'Service_Min': s['service_min'],
            'Miles': s['miles'],
        })
pd.DataFrame(schedule_rows).to_csv('fact_schedule.csv', index=False)

# ── Cost Summary ──────────────────────────────────────────────
cost_rows = [{
    'Day': d['day'], 'Day_Label': d['day_label'],
    'Total_Miles': d['total_miles'], 'Total_Hours': d['total_hours'],
    'Labor_Cost': d['labor_cost'], 'Truck_Cost': d['truck_cost'],
    'Holding_Cost': d['holding_cost'], 'Total_Cost': d['total_cost'],
    'CO2_kg': d['co2_kg'],
} for d in daily_details]
pd.DataFrame(cost_rows).to_csv('fact_cost_summary.csv', index=False)

# ── Loading Manifest ──────────────────────────────────────────
manifest_rows = []
for d in daily_details:
    load_order = list(reversed(d['stops']))
    for pos, s in enumerate(load_order, 1):
        cid = s['customer_id']
        items = orders_df[orders_df['Customer ID'] == cid]
        for _, item in items.iterrows():
            manifest_rows.append({
                'Day': d['day'], 'Day_Label': d['day_label'],
                'Load_Position': pos,
                'Delivery_Stop': len(d['stops']) - pos + 1,
                'Node_ID': s['node_id'], 'Customer_ID': cid,
                'Load_Label': s['load_label'], 'Address': s['address'],
                'Product': item['Product Description'],
                'Brand': item['Brand'],
                'Category': item['Category'],
                'Quantity': item['Quantity'],
                'CuFt': item['CuFt'], 'Lbs': item['Lbs'],
            })
pd.DataFrame(manifest_rows).to_csv('fact_loading_manifest.csv', index=False)

# ── CO₂ Comparison ────────────────────────────────────────────
pd.DataFrame([
    {'Type': 'Optimized',      'Miles': opt_miles,   'CO2_kg': opt_co2},
    {'Type': 'Naive Baseline', 'Miles': naive_miles, 'CO2_kg': naive_co2},
]).to_csv('fact_co2_comparison.csv', index=False)

print('Exported: fact_schedule.csv, fact_cost_summary.csv, fact_loading_manifest.csv, fact_co2_comparison.csv')
print('Use fact_ prefix for Power BI semantic model naming convention.')

Exported: fact_schedule.csv, fact_cost_summary.csv, fact_loading_manifest.csv, fact_co2_comparison.csv
Use fact_ prefix for Power BI semantic model naming convention.


## 13. Disruption Scenarios
Uncomment one block, then re-run from Cell 1c (Delivery Orders) onwards.

**For the live demo:** Simply edit the Status column in the Input Template on SharePoint,
then click Refresh in Power BI. The pipeline re-reads Active orders and recalculates everything.

In [20]:
# ── SCENARIO A: Customer not available ─────────────────────
# In the Input Template: set Status to 'Rescheduled' for all
# lines belonging to the unavailable customer, fill Reschedule Date.
# Then Refresh in Power BI → pipeline filters them out automatically.

# ── SCENARIO B: Partial order cancellation ─────────────────
# In the Input Template: set Status to 'Cancelled' for specific
# line numbers. Then Refresh → recalculates with reduced load.

# ── SCENARIO C: Vehicle breakdown ──────────────────────────
# Add a 'Delay_Min' column to Model Config for extra dispatch time.
# Pipeline reads it and adds to Day 1 cost/CO₂.

print('Disruption handling is built into the template Status column.')
print('Change Status → Cancelled or Rescheduled → Save → Refresh Power BI.')

Disruption handling is built into the template Status column.
Change Status → Cancelled or Rescheduled → Save → Refresh Power BI.
